In [1]:
"""
=============================================================
  CRYPTO STATISTICAL ARBITRAGE — PAPER TRADING BOT
  ─────────────────────────────────────────────────────────
  STRATEGY:
    Find crypto pairs that move together (cointegrated).
    When their spread diverges beyond 2 standard deviations,
    bet on reversion: long the underperformer, short the
    overperformer. Exit when spread returns to mean.

  PAIRS MONITORED (BTC/ETH universe):
    BTC/USDT  vs  ETH/USDT    (most correlated)
    BTC/USDT  vs  BNB/USDT
    ETH/USDT  vs  BNB/USDT
    SOL/USDT  vs  AVAX/USDT
    ETH/USDT  vs  SOL/USDT
    BTC/USDT  vs  SOL/USDT

  PAPER TRADING:
    Real prices from Binance WebSocket (free, no API key)
    Fake money — no real trades executed
    All P&L tracked and logged to Excel

  HOW IT WORKS (each tick):
    1. Fetch latest prices for all pairs
    2. Compute spread = log(price_A) - hedge_ratio × log(price_B)
    3. Compute z-score = (spread - mean) / std
    4. If z > +2 → SHORT A, LONG B  (A overvalued vs B)
    5. If z < -2 → LONG A, SHORT B  (A undervalued vs B)
    6. If |z| < 0.5 → close position (spread reverted)
    7. Stop loss if |z| > 3.5 (spread blowing out)
=============================================================
"""

import subprocess
subprocess.run([
    "pip", "install",
    "ccxt", "pandas", "numpy", "openpyxl",
    "websocket-client", "requests", "-q"
], check=False)

import ccxt
import pandas as pd
import numpy as np
import time, os, json, threading, logging
from datetime import datetime
from collections import deque

# ═══════════════════════════════════════════════════════════
#  SETTINGS
# ═══════════════════════════════════════════════════════════
PAPER_CAPITAL        = 10_000      # fake USDT to trade with
ALLOCATION_PER_PAIR  = 0.20        # 20% of capital per pair max
MAX_OPEN_PAIRS       = 3           # max simultaneous positions

ENTRY_ZSCORE         = 2.0         # enter when z-score exceeds this
EXIT_ZSCORE          = 0.5         # exit when z-score returns below this
STOP_ZSCORE          = 3.5         # stop loss if spread blows out
LOOKBACK             = 100         # bars to compute rolling mean/std
MIN_CORRELATION      = 0.80        # minimum correlation to trade a pair
UPDATE_INTERVAL_SEC  = 10          # fetch prices every 10 seconds

LOG_FILE             = "crypto_arb_log.csv"
EXCEL_FILE           = "crypto_arb_results.xlsx"
STATE_FILE           = "arb_state.json"

# Pairs to monitor
PAIRS = [
    ("BTC/USDT",  "ETH/USDT"),
    ("BTC/USDT",  "BNB/USDT"),
    ("ETH/USDT",  "BNB/USDT"),
    ("ETH/USDT",  "SOL/USDT"),
    ("SOL/USDT",  "AVAX/USDT"),
    ("BTC/USDT",  "SOL/USDT"),
]

# ═══════════════════════════════════════════════════════════
#  LOGGING
# ═══════════════════════════════════════════════════════════
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)s  %(message)s",
    datefmt = "%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("arb_bot.log"),
    ]
)
log = logging.getLogger("ArBot")

# ═══════════════════════════════════════════════════════════
#  EXCHANGE  (Binance, no API key needed for public data)
# ═══════════════════════════════════════════════════════════
def init_exchange():
    exchange = ccxt.binance({
        "enableRateLimit": True,
        "options": {"defaultType": "spot"},
    })
    log.info("Exchange initialised: Binance (public data only)")
    return exchange

# ═══════════════════════════════════════════════════════════
#  PRICE HISTORY  — bootstrap with recent candles
# ═══════════════════════════════════════════════════════════
def fetch_history(exchange, symbol, limit=200):
    """Fetch recent 1-hour OHLCV bars to initialise spread stats."""
    try:
        ohlcv = exchange.fetch_ohlcv(symbol, "1h", limit=limit)
        df    = pd.DataFrame(ohlcv, columns=["ts","open","high","low","close","vol"])
        df["ts"] = pd.to_datetime(df["ts"], unit="ms")
        df.set_index("ts", inplace=True)
        return df["close"]
    except Exception as e:
        log.error(f"History fetch failed for {symbol}: {e}")
        return None

def fetch_price(exchange, symbol):
    """Fetch latest ticker price."""
    try:
        ticker = exchange.fetch_ticker(symbol)
        return float(ticker["last"])
    except Exception as e:
        log.error(f"Price fetch failed for {symbol}: {e}")
        return None

# ═══════════════════════════════════════════════════════════
#  SPREAD STATISTICS
# ═══════════════════════════════════════════════════════════
def compute_hedge_ratio(log_price_a, log_price_b):
    """
    OLS regression: log(A) = hedge_ratio × log(B) + alpha
    hedge_ratio tells us how many units of B to hold per unit of A.
    """
    from numpy.linalg import lstsq
    X   = np.column_stack([log_price_b, np.ones(len(log_price_b))])
    res = lstsq(X, log_price_a, rcond=None)
    return float(res[0][0])  # hedge ratio (beta)

def compute_spread(log_a, log_b, hedge_ratio):
    """Spread = log(A) - hedge_ratio × log(B)"""
    return log_a - hedge_ratio * log_b

def compute_zscore(spread_series):
    """Z-score of latest spread value vs rolling window."""
    mean = spread_series.rolling(LOOKBACK).mean()
    std  = spread_series.rolling(LOOKBACK).std()
    z    = (spread_series - mean) / std
    return z

def correlation(series_a, series_b):
    """Pearson correlation of log returns."""
    ret_a = np.log(series_a).diff().dropna()
    ret_b = np.log(series_b).diff().dropna()
    n     = min(len(ret_a), len(ret_b))
    return float(np.corrcoef(ret_a.tail(n), ret_b.tail(n))[0,1])

# ═══════════════════════════════════════════════════════════
#  PAPER PORTFOLIO
# ═══════════════════════════════════════════════════════════
class PaperPortfolio:
    def __init__(self, capital):
        self.capital        = capital
        self.cash           = capital
        self.positions      = {}    # pair_key → position dict
        self.trade_log      = []
        self.equity_history = []

    def open_position(self, pair_key, sym_a, sym_b,
                      price_a, price_b, hedge_ratio,
                      direction, zscore, allocation):
        """
        direction = "LONG_SPREAD"  → long A, short B
        direction = "SHORT_SPREAD" → short A, long B
        """
        if pair_key in self.positions:
            return False  # already in this pair

        if self.cash < allocation:
            log.warning(f"Not enough cash for {pair_key}")
            return False

        units_a = (allocation * 0.5) / price_a
        units_b = (allocation * 0.5) / price_b * hedge_ratio

        self.cash -= allocation
        self.positions[pair_key] = {
            "sym_a":        sym_a,
            "sym_b":        sym_b,
            "direction":    direction,
            "entry_price_a":price_a,
            "entry_price_b":price_b,
            "units_a":      units_a,
            "units_b":      units_b,
            "hedge_ratio":  hedge_ratio,
            "allocation":   allocation,
            "entry_zscore": zscore,
            "entry_time":   datetime.now(),
        }
        log.info(
            f"📈 OPENED  {pair_key}  {direction}  "
            f"z={zscore:.2f}  "
            f"A={price_a:.4f}  B={price_b:.4f}  "
            f"alloc=${allocation:,.0f}"
        )
        return True

    def close_position(self, pair_key, price_a, price_b,
                       zscore, reason):
        if pair_key not in self.positions:
            return None

        pos     = self.positions[pair_key]
        dir_    = pos["direction"]

        # P&L calculation
        if dir_ == "LONG_SPREAD":
            # Long A → profit if A went up
            # Short B → profit if B went down
            pnl_a = (price_a - pos["entry_price_a"]) * pos["units_a"]
            pnl_b = (pos["entry_price_b"] - price_b) * pos["units_b"]
        else:
            # Short A → profit if A went down
            # Long B → profit if B went up
            pnl_a = (pos["entry_price_a"] - price_a) * pos["units_a"]
            pnl_b = (price_b - pos["entry_price_b"]) * pos["units_b"]

        total_pnl    = pnl_a + pnl_b
        self.cash   += pos["allocation"] + total_pnl
        hold_mins    = (datetime.now() - pos["entry_time"]).seconds // 60

        trade = {
            "Pair":           pair_key,
            "Direction":      dir_,
            "Entry Time":     pos["entry_time"].strftime("%Y-%m-%d %H:%M"),
            "Exit Time":      datetime.now().strftime("%Y-%m-%d %H:%M"),
            "Hold (mins)":    hold_mins,
            "Entry Z-score":  round(pos["entry_zscore"], 3),
            "Exit Z-score":   round(zscore, 3),
            "Entry Price A":  pos["entry_price_a"],
            "Entry Price B":  pos["entry_price_b"],
            "Exit Price A":   price_a,
            "Exit Price B":   price_b,
            "PnL A ($)":      round(pnl_a, 4),
            "PnL B ($)":      round(pnl_b, 4),
            "Total PnL ($)":  round(total_pnl, 4),
            "Return %":       round(total_pnl / pos["allocation"] * 100, 3),
            "Exit Reason":    reason,
            "Win":            total_pnl > 0,
        }
        self.trade_log.append(trade)
        del self.positions[pair_key]

        emoji = "✅" if total_pnl > 0 else "❌"
        log.info(
            f"{emoji} CLOSED  {pair_key}  {reason}  "
            f"z={zscore:.2f}  "
            f"PnL=${total_pnl:+.2f}  "
            f"({trade['Return %']:+.2f}%)"
        )
        return trade

    def mark_to_market(self, prices):
        """Compute current portfolio value."""
        value = self.cash
        for key, pos in self.positions.items():
            p_a = prices.get(pos["sym_a"])
            p_b = prices.get(pos["sym_b"])
            if p_a is None or p_b is None:
                continue
            if pos["direction"] == "LONG_SPREAD":
                pnl_a = (p_a - pos["entry_price_a"]) * pos["units_a"]
                pnl_b = (pos["entry_price_b"] - p_b) * pos["units_b"]
            else:
                pnl_a = (pos["entry_price_a"] - p_a) * pos["units_a"]
                pnl_b = (p_b - pos["entry_price_b"]) * pos["units_b"]
            value += pos["allocation"] + pnl_a + pnl_b
        return value

    def stats(self):
        if not self.trade_log:
            return {}
        df   = pd.DataFrame(self.trade_log)
        wins = df[df["Win"]]
        loss = df[~df["Win"]]
        return {
            "Total Trades":    len(df),
            "Win Rate %":      round(len(wins)/len(df)*100, 2),
            "Total PnL $":     round(df["Total PnL ($)"].sum(), 2),
            "Avg Win $":       round(wins["Total PnL ($)"].mean(), 2) if len(wins) > 0 else 0,
            "Avg Loss $":      round(loss["Total PnL ($)"].mean(), 2) if len(loss) > 0 else 0,
            "Best Trade $":    round(df["Total PnL ($)"].max(), 2),
            "Worst Trade $":   round(df["Total PnL ($)"].min(), 2),
            "Avg Hold (mins)": round(df["Hold (mins)"].mean(), 1),
        }

# ═══════════════════════════════════════════════════════════
#  PAIR STATE — tracks rolling spread stats per pair
# ═══════════════════════════════════════════════════════════
class PairState:
    def __init__(self, sym_a, sym_b):
        self.sym_a       = sym_a
        self.sym_b       = sym_b
        self.key         = f"{sym_a.replace('/','')}__{sym_b.replace('/','')}"
        self.prices_a    = deque(maxlen=300)
        self.prices_b    = deque(maxlen=300)
        self.hedge_ratio = None
        self.corr        = None
        self.zscore      = None
        self.spread      = None
        self.ready       = False

    def update(self, price_a, price_b):
        self.prices_a.append(price_a)
        self.prices_b.append(price_b)

        if len(self.prices_a) < LOOKBACK + 10:
            return  # not enough data yet

        pa  = np.array(self.prices_a)
        pb  = np.array(self.prices_b)
        la  = np.log(pa)
        lb  = np.log(pb)

        # Update hedge ratio every 50 ticks
        if len(self.prices_a) % 50 == 0 or self.hedge_ratio is None:
            self.hedge_ratio = compute_hedge_ratio(la, lb)
            self.corr        = float(np.corrcoef(
                np.diff(la[-100:]), np.diff(lb[-100:])
            )[0,1])

        if self.hedge_ratio is None:
            return

        spread_series = pd.Series(la - self.hedge_ratio * lb)
        z_series      = compute_zscore(spread_series)

        self.spread = float(spread_series.iloc[-1])
        self.zscore = float(z_series.iloc[-1]) if not pd.isna(z_series.iloc[-1]) else None
        self.ready  = True

# ═══════════════════════════════════════════════════════════
#  SAVE / LOAD STATE
# ═══════════════════════════════════════════════════════════
def save_state(portfolio):
    state = {
        "cash":       portfolio.cash,
        "trade_count":len(portfolio.trade_log),
        "equity":     portfolio.equity_history[-1] if portfolio.equity_history else portfolio.cash,
        "timestamp":  datetime.now().isoformat(),
    }
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

def save_trades_excel(portfolio):
    if not portfolio.trade_log:
        return
    trades = pd.DataFrame(portfolio.trade_log)
    eq_df  = pd.DataFrame(portfolio.equity_history,
                          columns=["timestamp","equity"])
    s      = portfolio.stats()

    with pd.ExcelWriter(EXCEL_FILE, engine="openpyxl") as writer:
        trades.to_excel(writer, sheet_name="All Trades", index=False)

        if len(trades) > 0:
            wins = trades[trades["Win"]]
            loss = trades[~trades["Win"]]
            if len(wins) > 0:
                wins.to_excel(writer, sheet_name="Winning Trades", index=False)
            if len(loss) > 0:
                loss.to_excel(writer, sheet_name="Losing Trades", index=False)

        eq_df.to_excel(writer, sheet_name="Equity Curve", index=False)

        pd.DataFrame([s]).T.reset_index().rename(
            columns={"index":"Metric", 0:"Value"}
        ).to_excel(writer, sheet_name="Summary", index=False)

    log.info(f"💾 Trades saved → {EXCEL_FILE}")

def append_csv(trade):
    """Append single trade to CSV for real-time log."""
    df = pd.DataFrame([trade])
    df.to_csv(LOG_FILE, mode="a",
              header=not os.path.exists(LOG_FILE),
              index=False)

# ═══════════════════════════════════════════════════════════
#  DASHBOARD  — prints to terminal every cycle
# ═══════════════════════════════════════════════════════════
def print_dashboard(portfolio, pair_states, prices, cycle):
    os.system("cls" if os.name=="nt" else "clear")
    equity = portfolio.mark_to_market(prices)
    pnl    = equity - PAPER_CAPITAL
    pnl_pct= pnl / PAPER_CAPITAL * 100
    stats  = portfolio.stats()

    print("╔══════════════════════════════════════════════════════════╗")
    print("║  ♟ CRYPTO STAT ARB — PAPER TRADING BOT                  ║")
    print(f"║  Cycle #{cycle:>5}  |  {datetime.now().strftime('%H:%M:%S')}                          ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print(f"║  💰 Capital       : ${PAPER_CAPITAL:>10,.2f}                      ║")
    print(f"║  📈 Current equity: ${equity:>10,.2f}  ({pnl_pct:+.2f}%)           ║")
    print(f"║  💵 Cash          : ${portfolio.cash:>10,.2f}                      ║")
    print(f"║  📊 Open positions: {len(portfolio.positions):>2}                               ║")
    print(f"║  🔄 Total trades  : {stats.get('Total Trades',0):>3}  |  Win rate: {stats.get('Win Rate %',0):>5.1f}%          ║")
    print(f"║  💸 Total PnL     : ${stats.get('Total PnL $',0):>+10.2f}                     ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print("║  PAIR SPREADS                                            ║")
    print("║  Pair                    Z-Score  Corr   Status          ║")
    print("╠══════════════════════════════════════════════════════════╣")

    for ps in pair_states:
        if not ps.ready or ps.zscore is None:
            status = "⏳ warming up"
            z_str  = "  —  "
            c_str  = "  —  "
        else:
            z      = ps.zscore
            z_str  = f"{z:+.3f}"
            c_str  = f"{ps.corr:.3f}" if ps.corr else "—"
            if abs(z) > STOP_ZSCORE:
                status = "🚨 STOP ZONE"
            elif z > ENTRY_ZSCORE:
                status = "📉 SHORT SPREAD"
            elif z < -ENTRY_ZSCORE:
                status = "📈 LONG SPREAD"
            elif abs(z) < EXIT_ZSCORE and ps.key in portfolio.positions:
                status = "🔄 EXIT ZONE"
            elif ps.key in portfolio.positions:
                pos = portfolio.positions[ps.key]
                curr_pnl = 0
                p_a = prices.get(ps.sym_a)
                p_b = prices.get(ps.sym_b)
                if p_a and p_b:
                    if pos["direction"] == "LONG_SPREAD":
                        curr_pnl = ((p_a - pos["entry_price_a"]) * pos["units_a"] +
                                    (pos["entry_price_b"] - p_b) * pos["units_b"])
                    else:
                        curr_pnl = ((pos["entry_price_a"] - p_a) * pos["units_a"] +
                                    (p_b - pos["entry_price_b"]) * pos["units_b"])
                status = f"{'✅' if curr_pnl>=0 else '❌'} OPEN PnL ${curr_pnl:+.2f}"
            elif ps.corr and ps.corr < MIN_CORRELATION:
                status = "⚠️  low corr"
            else:
                status = "✅ monitoring"

        pair_str = f"{ps.sym_a[:8]}/{ps.sym_b[:8]}"
        print(f"║  {pair_str:<22}  {z_str:<8} {c_str:<7}  {status:<18}  ║")

    print("╠══════════════════════════════════════════════════════════╣")

    # Recent trades
    if portfolio.trade_log:
        print("║  RECENT TRADES                                           ║")
        for t in portfolio.trade_log[-3:]:
            emoji  = "✅" if t["Win"] else "❌"
            pnl_s  = f"${t['Total PnL ($)']:+.2f}"
            print(f"║  {emoji} {t['Pair'][:28]:<28} {pnl_s:>8}  {t['Exit Reason'][:10]:<10}  ║")

    print("╚══════════════════════════════════════════════════════════╝")
    print(f"  Updates every {UPDATE_INTERVAL_SEC}s  |  Ctrl+C to stop & save results")

# ═══════════════════════════════════════════════════════════
#  MAIN BOT LOOP
# ═══════════════════════════════════════════════════════════
def run_bot():
    log.info("Starting Crypto Stat Arb Paper Trading Bot...")

    exchange    = init_exchange()
    portfolio   = PaperPortfolio(PAPER_CAPITAL)
    pair_states = [PairState(a, b) for a, b in PAIRS]

    # Bootstrap price history
    log.info("Bootstrapping price history (this takes ~30 seconds)...")
    all_symbols = list(set(s for pair in PAIRS for s in pair))

    for ps in pair_states:
        log.info(f"   Fetching history: {ps.sym_a}, {ps.sym_b}")
        hist_a = fetch_history(exchange, ps.sym_a, limit=250)
        hist_b = fetch_history(exchange, ps.sym_b, limit=250)
        if hist_a is not None and hist_b is not None:
            common = hist_a.index.intersection(hist_b.index)
            for ts in common:
                ps.update(float(hist_a[ts]), float(hist_b[ts]))
        time.sleep(0.5)  # rate limit

    log.info("Bootstrap complete. Starting live monitoring...\n")

    cycle          = 0
    save_interval  = 30   # save Excel every 30 cycles

    while True:
        try:
            cycle += 1

            # 1. Fetch current prices
            prices = {}
            for sym in all_symbols:
                p = fetch_price(exchange, sym)
                if p: prices[sym] = p
            time.sleep(1)  # brief pause after fetching

            if len(prices) < 2:
                log.warning("Not enough prices fetched, skipping cycle")
                time.sleep(UPDATE_INTERVAL_SEC)
                continue

            # 2. Update pair states
            for ps in pair_states:
                p_a = prices.get(ps.sym_a)
                p_b = prices.get(ps.sym_b)
                if p_a and p_b:
                    ps.update(p_a, p_b)

            # 3. Check exits first
            for ps in pair_states:
                if not ps.ready or ps.zscore is None: continue
                if ps.key not in portfolio.positions: continue

                p_a = prices.get(ps.sym_a)
                p_b = prices.get(ps.sym_b)
                if not p_a or not p_b: continue

                z = ps.zscore

                if abs(z) <= EXIT_ZSCORE:
                    trade = portfolio.close_position(
                        ps.key, p_a, p_b, z, "Mean Reversion"
                    )
                    if trade: append_csv(trade)

                elif abs(z) >= STOP_ZSCORE:
                    trade = portfolio.close_position(
                        ps.key, p_a, p_b, z, "Stop Loss"
                    )
                    if trade: append_csv(trade)

            # 4. Check entries
            if len(portfolio.positions) < MAX_OPEN_PAIRS:
                for ps in pair_states:
                    if not ps.ready or ps.zscore is None: continue
                    if ps.key in portfolio.positions: continue
                    if ps.corr and ps.corr < MIN_CORRELATION: continue

                    p_a = prices.get(ps.sym_a)
                    p_b = prices.get(ps.sym_b)
                    if not p_a or not p_b: continue

                    z          = ps.zscore
                    allocation = PAPER_CAPITAL * ALLOCATION_PER_PAIR

                    if z > ENTRY_ZSCORE:
                        # A overvalued → short A, long B
                        portfolio.open_position(
                            ps.key, ps.sym_a, ps.sym_b,
                            p_a, p_b, ps.hedge_ratio,
                            "SHORT_SPREAD", z, allocation
                        )
                    elif z < -ENTRY_ZSCORE:
                        # A undervalued → long A, short B
                        portfolio.open_position(
                            ps.key, ps.sym_a, ps.sym_b,
                            p_a, p_b, ps.hedge_ratio,
                            "LONG_SPREAD", z, allocation
                        )

            # 5. Track equity
            equity = portfolio.mark_to_market(prices)
            portfolio.equity_history.append(
                (datetime.now().strftime("%Y-%m-%d %H:%M:%S"), equity)
            )

            # 6. Dashboard
            print_dashboard(portfolio, pair_states, prices, cycle)

            # 7. Periodic save
            if cycle % save_interval == 0:
                save_trades_excel(portfolio)
                save_state(portfolio)

            time.sleep(UPDATE_INTERVAL_SEC)

        except KeyboardInterrupt:
            log.info("\nStopping bot — saving results...")
            save_trades_excel(portfolio)
            save_state(portfolio)

            s = portfolio.stats()
            print("\n" + "="*55)
            print("  FINAL PAPER TRADING RESULTS")
            print("="*55)
            equity = portfolio.mark_to_market(prices)
            print(f"  Starting Capital : ${PAPER_CAPITAL:,.2f}")
            print(f"  Final Equity     : ${equity:,.2f}")
            print(f"  Total PnL        : ${equity-PAPER_CAPITAL:+,.2f} ({(equity-PAPER_CAPITAL)/PAPER_CAPITAL*100:+.2f}%)")
            for k, v in s.items():
                print(f"  {k:<22} : {v}")
            print(f"\n  Results saved → {EXCEL_FILE}")
            print("="*55)
            break

        except Exception as e:
            log.error(f"Cycle error: {e}")
            time.sleep(UPDATE_INTERVAL_SEC)

# ═══════════════════════════════════════════════════════════
#  ENTRY POINT
# ═══════════════════════════════════════════════════════════
if __name__ == "__main__":
    print("\n" + "="*60)
    print("  ♟ CRYPTO STAT ARB — PAPER TRADING BOT")
    print("  Strategy : Statistical arbitrage on correlated pairs")
    print("  Capital  : ${:,.0f} (PAPER — no real money)".format(PAPER_CAPITAL))
    print("  Pairs    : {} pairs monitored".format(len(PAIRS)))
    print("  Entry    : Z-score > {:.1f} standard deviations".format(ENTRY_ZSCORE))
    print("  Exit     : Z-score < {:.1f} (mean reversion)".format(EXIT_ZSCORE))
    print("  Stop     : Z-score > {:.1f} (spread blow-out)".format(STOP_ZSCORE))
    print("="*60)
    print("\nPress Ctrl+C at any time to stop and save results\n")
    time.sleep(2)
    run_bot()


  ♟ CRYPTO STAT ARB — PAPER TRADING BOT
  Strategy : Statistical arbitrage on correlated pairs
  Capital  : $10,000 (PAPER — no real money)
  Pairs    : 6 pairs monitored
  Entry    : Z-score > 2.0 standard deviations
  Exit     : Z-score < 0.5 (mean reversion)
  Stop     : Z-score > 3.5 (spread blow-out)

Press Ctrl+C at any time to stop and save results



11:54:45  INFO  Starting Crypto Stat Arb Paper Trading Bot...
11:54:45  INFO  Exchange initialised: Binance (public data only)
11:54:45  INFO  Bootstrapping price history (this takes ~30 seconds)...
11:54:45  INFO     Fetching history: BTC/USDT, ETH/USDT
11:54:51  INFO     Fetching history: BTC/USDT, BNB/USDT
11:54:52  INFO     Fetching history: ETH/USDT, BNB/USDT
11:54:53  INFO     Fetching history: ETH/USDT, SOL/USDT
11:54:54  INFO     Fetching history: SOL/USDT, AVAX/USDT
11:54:55  INFO     Fetching history: BTC/USDT, SOL/USDT
11:54:57  INFO  Bootstrap complete. Starting live monitoring...

11:54:59  INFO  📈 OPENED  BTCUSDT__BNBUSDT  LONG_SPREAD  z=-2.02  A=77442.9700  B=656.7600  alloc=$2,000
--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Pytho

╔══════════════════════════════════════════════════════════╗
║  ♟ CRYPTO STAT ARB — PAPER TRADING BOT                  ║
║  Cycle #    1  |  11:54:59                          ║
╠══════════════════════════════════════════════════════════╣
║  💰 Capital       : $ 10,000.00                      ║
║  📈 Current equity: $ 10,000.00  (+0.00%)           ║
║  💵 Cash          : $  6,000.00                      ║
║  📊 Open positions:  2                               ║
║  🔄 Total trades  :   0  |  Win rate:   0.0%          ║
║  💸 Total PnL     : $     +0.00                     ║
╠══════════════════════════════════════════════════════════╣
║  PAIR SPREADS                                            ║
║  Pair                    Z-Score  Corr   Status          ║
╠══════════════════════════════════════════════════════════╣
║  BTC/USDT/ETH/USDT       +0.804   0.903    ✅ monitoring        ║
║  BTC/USDT/BNB/USDT       -2.020   0.866    📈 LONG SPREAD       ║
║  ETH/USDT/BNB/USDT       -1.894   0.850    ✅ mo

12:05:56  INFO  ✅ CLOSED  SOLUSDT__AVAXUSDT  Mean Reversion  z=-0.49  PnL=$+2.97  (+0.15%)
--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 16: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\USER\AppData\Roaming\Python\P

╔══════════════════════════════════════════════════════════╗
║  ♟ CRYPTO STAT ARB — PAPER TRADING BOT                  ║
║  Cycle #   55  |  12:05:56                          ║
╠══════════════════════════════════════════════════════════╣
║  💰 Capital       : $ 10,000.00                      ║
║  📈 Current equity: $ 10,002.79  (+0.03%)           ║
║  💵 Cash          : $  8,002.97                      ║
║  📊 Open positions:  1                               ║
║  🔄 Total trades  :   1  |  Win rate: 100.0%          ║
║  💸 Total PnL     : $     +2.97                     ║
╠══════════════════════════════════════════════════════════╣
║  PAIR SPREADS                                            ║
║  Pair                    Z-Score  Corr   Status          ║
╠══════════════════════════════════════════════════════════╣
║  BTC/USDT/ETH/USDT       -0.159   0.924    ✅ monitoring        ║
║  BTC/USDT/BNB/USDT       -0.637   0.889    ❌ OPEN PnL $-0.18   ║
║  ETH/USDT/BNB/USDT       -0.606   0.882    ✅ mo

...

╔══════════════════════════════════════════════════════════╗
║  ♟ CRYPTO STAT ARB — PAPER TRADING BOT                  ║
║  Cycle #  149  |  12:25:13                          ║
╠══════════════════════════════════════════════════════════╣
║  💰 Capital       : $ 10,000.00                      ║
║  📈 Current equity: $ 10,002.50  (+0.03%)           ║
║  💵 Cash          : $ 10,002.50                      ║
║  📊 Open positions:  0                               ║
║  🔄 Total trades  :   4  |  Win rate:  25.0%          ║
║  💸 Total PnL     : $     +2.50                     ║
╠══════════════════════════════════════════════════════════╣
║  PAIR SPREADS                                            ║
║  Pair                    Z-Score  Corr   Status          ║
╠══════════════════════════════════════════════════════════╣
║  BTC/USDT/ETH/USDT       +0.409   0.825    ✅ monitoring        ║
║  BTC/USDT/BNB/USDT       -0.240   0.520    ⚠️  low corr        ║
║  ETH/USDT/BNB/USDT       -2.372   0.619    📈 LO

12:25:25  INFO  💾 Trades saved → crypto_arb_results.xlsx
--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f4be' in position 16: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\traitle

╔══════════════════════════════════════════════════════════╗
║  ♟ CRYPTO STAT ARB — PAPER TRADING BOT                  ║
║  Cycle #  151  |  12:25:38                          ║
╠══════════════════════════════════════════════════════════╣
║  💰 Capital       : $ 10,000.00                      ║
║  📈 Current equity: $ 10,002.50  (+0.03%)           ║
║  💵 Cash          : $ 10,002.50                      ║
║  📊 Open positions:  0                               ║
║  🔄 Total trades  :   4  |  Win rate:  25.0%          ║
║  💸 Total PnL     : $     +2.50                     ║
╠══════════════════════════════════════════════════════════╣
║  PAIR SPREADS                                            ║
║  Pair                    Z-Score  Corr   Status          ║
╠══════════════════════════════════════════════════════════╣
║  BTC/USDT/ETH/USDT       +0.390   0.834    ✅ monitoring        ║
║  BTC/USDT/BNB/USDT       -0.019   0.519    ⚠️  low corr        ║
║  ETH/USDT/BNB/USDT       -1.821   0.616    ⚠️  

12:28:02  INFO  
Stopping bot — saving results...
12:28:02  INFO  💾 Trades saved → crypto_arb_results.xlsx
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_9436\724486743.py", line 582, in run_bot
    time.sleep(UPDATE_INTERVAL_SEC)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f4be' in position 16: character maps to <undefined>



  FINAL PAPER TRADING RESULTS
  Starting Capital : $10,000.00
  Final Equity     : $10,002.50
  Total PnL        : $+2.50 (+0.03%)
  Total Trades           : 4
  Win Rate %             : 25.0
  Total PnL $            : 2.5
  Avg Win $              : 2.97
  Avg Loss $             : -0.15
  Best Trade $           : 2.97
  Worst Trade $          : -0.42
  Avg Hold (mins)        : 7.8

  Results saved → crypto_arb_results.xlsx
